# Loan Scoring Pipeline

This notebook builds an end-to-end loan scoring pipeline using `decider`.
Two modules are composed with `|` into a single sequential pipeline:

- **IncomeEnricher** — derives `debt_to_income` and `monthly_surplus` from raw applicant data
- **RiskScorer** — assigns a `risk_score` and `risk_tier` using a configurable threshold

We also demonstrate config serialisation and how to hot-swap a tighter threshold.

In [ ]:
%pip install -q decider


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import polars as pl

In [ ]:
from decider.modules.functional import generate_from_functions
from decider.modules._ext import register_graph_module, GraphModule
from pydantic import BaseModel

## Sample data

Five loan applicants with income, monthly debt obligations, and requested loan amount.

In [ ]:
applicants = pl.DataFrame({
    "customer_id":   ["C001", "C002", "C003", "C004", "C005"],
    "income":        [50_000, 35_000, 80_000, 22_000, 60_000],
    "monthly_debt":  [1_200,    900,  2_100,    700,  1_500],
    "loan_amount":   [10_000, 15_000, 25_000,  5_000, 20_000],
})

print(applicants)

## Module 1 — IncomeEnricher

Derives two financial ratios from the raw columns.

In [ ]:
def debt_to_income(monthly_debt: pl.Expr, income: pl.Expr) -> pl.Expr:
    return (monthly_debt * 12) / income

def monthly_surplus(income: pl.Expr, monthly_debt: pl.Expr) -> pl.Expr:
    return income / 12 - monthly_debt

IncomeEnricher = generate_from_functions("income_enricher", debt_to_income, monthly_surplus)
register_graph_module(IncomeEnricher)
income_enricher = IncomeEnricher(name="income_enricher")

print(income_enricher({"input": applicants}))

## Module 2 — RiskScorer

Computes a `risk_score` from the enriched columns and assigns a `risk_tier`.
The threshold that separates high from low risk is injected from `RiskConfig`.

In [ ]:
class RiskConfig(BaseModel):
    threshold: float = 0.4

def risk_score(debt_to_income: pl.Expr, monthly_surplus: pl.Expr) -> pl.Expr:
    return (debt_to_income * 0.7 + (1 / (monthly_surplus + 1)) * 0.3).round(4)

def risk_tier(risk_score: pl.Expr, config: RiskConfig) -> pl.Expr:
    return pl.when(risk_score > config.threshold).then(pl.lit("HIGH")).otherwise(pl.lit("LOW"))

RiskScorer = generate_from_functions("risk_scorer", risk_score, risk_tier)
register_graph_module(RiskScorer)
risk_scorer = RiskScorer(name="risk_scorer", threshold=0.4)

print(risk_scorer)

## Composing the pipeline

The `|` operator chains `IncomeEnricher` and `RiskScorer` into a `SequentialModule`.
Each step receives the previous step's output as its `"input"` frame.

In [ ]:
pipeline = income_enricher | risk_scorer
print(type(pipeline).__name__, "—", pipeline.name)

result = pipeline({"input": applicants})
print(result.select(["customer_id", "debt_to_income", "monthly_surplus", "risk_score", "risk_tier"]))

## Config roundtrip

`model_dump()` serialises the full pipeline (including nested configs) to a plain dict.
`GraphModule.model_validate()` reconstructs a live module from that dict.

In [ ]:
import json

cfg = pipeline.model_dump()
print(json.dumps(cfg, indent=2))

In [ ]:
restored = GraphModule.model_validate(cfg).root
restored_result = restored({"input": applicants})
print("Roundtrip output matches original:", result.equals(restored_result))
print(restored_result.select(["customer_id", "risk_score", "risk_tier"]))

## Tighter threshold

Swap in a stricter `threshold=0.3` to see how the `risk_tier` assignments change.

In [ ]:
strict_scorer = RiskScorer(name="risk_scorer", threshold=0.3)
strict_pipeline = income_enricher | strict_scorer

strict_result = strict_pipeline({"input": applicants})
comparison = result.select(["customer_id", "risk_tier"]).rename({"risk_tier": "tier_0.4"}).with_columns(
    strict_result["risk_tier"].alias("tier_0.3")
)
print(comparison)